# 00 — Warmup: writing an autoregressive decoding loop by hand

**Phase 0, exercise 1.** Goal: load Pythia-160M, generate ~50 tokens *without* using `model.generate()`, and convince yourself you understand what's happening at every step.

By the end of this notebook you should be able to answer:
1. What does the model output at each step, and why does it have that shape?
2. Why is the loop sequential — what data dependency forces it?
3. What is a KV cache and why does it speed things up? (We'll measure in notebook 01.)

**Compute:** runs fine on CPU; T4 makes it slightly faster but not necessary.

## Setup

> **Colab gotcha:** do **not** `pip install torch` in this notebook. Colab ships with torch already loaded into the kernel; reinstalling on top of a live import gets you a `partially initialized module 'torch'` circular-import crash. If you hit that, fix is: **Runtime → Restart session**, then re-run from the top *without* reinstalling torch. The setup cell below assumes torch and transformers are present (they are, on Colab).

In [ ]:
# Colab ships torch + transformers pre-installed — DO NOT reinstall torch in-session.
# If a later import fails because transformers is too old, upgrade only that, then
# restart the runtime before re-running this cell:
#     !pip install -q -U transformers
#     # then Runtime -> Restart session

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "EleutherAI/pythia-160m"
MODEL_REVISION = "step143000"  # the final checkpoint — pin by step for reproducibility
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0

torch.manual_seed(SEED)
print(f"torch {torch.__version__} | device: {DEVICE}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, revision=MODEL_REVISION).to(DEVICE)
model.eval()
print(f"loaded {MODEL_NAME} @ {MODEL_REVISION} | {sum(p.numel() for p in model.parameters()):,} params")

## What does one forward pass look like?

Take a prompt, tokenize it, push it through the model, look at the output.

Things to notice:
- `input_ids` has shape `[batch, sequence_length]`.
- `logits` has shape `[batch, sequence_length, vocab_size]`. **One distribution per input position** — the model predicts the *next* token at every position simultaneously during a forward pass. (This is why training is parallel.)
- For decoding, we only care about the **last** position's logits: that's the prediction for the token *after* the prompt.

In [ ]:
prompt = "The capital of France is"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)

with torch.no_grad():
    outputs = model(input_ids)

logits = outputs.logits  # [batch=1, seq_len, vocab_size]
print(f"input_ids shape: {input_ids.shape}")
print(f"logits shape:    {logits.shape}")

# look at the top-5 predictions for the next token
next_token_logits = logits[0, -1]  # last position
probs = F.softmax(next_token_logits, dim=-1)
top5_probs, top5_ids = probs.topk(5)
for prob, tok_id in zip(top5_probs.tolist(), top5_ids.tolist()):
    print(f"  {prob:.4f}  '{tokenizer.decode([tok_id])}'")

## The decoding loop (greedy, no cache)

Here is the simplest possible autoregressive decoder:

1. Run forward pass on the full sequence so far.
2. Take the last position's logits.
3. argmax → next token.
4. Append. Repeat.

This is **horribly inefficient** because step 1 reprocesses the entire sequence on every iteration. Notebook 01 fixes that with a KV cache. For now, the inefficiency is pedagogically useful: it makes the sequential data dependency obvious.

In [ ]:
def greedy_decode_naive(model, tokenizer, prompt: str, max_new_tokens: int = 50, verbose: bool = False) -> str:
    """Hand-rolled greedy decoding. No KV cache. No batching. No tricks."""
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    
    for step in range(max_new_tokens):
        with torch.no_grad():
            # *** the sequential bottleneck: we have to wait for step's output before we can run step+1 ***
            outputs = model(input_ids)
        
        next_token_logits = outputs.logits[0, -1]  # last position only
        next_token_id = next_token_logits.argmax(dim=-1).unsqueeze(0).unsqueeze(0)  # [1, 1]
        input_ids = torch.cat([input_ids, next_token_id], dim=1)
        
        if verbose:
            print(f"step {step:2d}: appended '{tokenizer.decode(next_token_id[0])}'")
        
        # stop on EOS
        if next_token_id.item() == tokenizer.eos_token_id:
            break
    
    return tokenizer.decode(input_ids[0])


output = greedy_decode_naive(model, tokenizer, prompt, max_new_tokens=30, verbose=False)
print(output)

## Logging the top-5 candidates at every step

This is the first peek at what the project is *really* about: at every position, how confident is the model in its top choice? If the top-1 has probability 0.95, the model basically already knows the answer — could we have predicted that without running the full forward pass?

(This is the seed of the parallel-safety question. We'll formalize it in Phase 1.)

In [ ]:
def greedy_decode_with_topk_log(model, tokenizer, prompt: str, max_new_tokens: int = 20, k: int = 5):
    """Greedy decode and record the top-k candidates at each step."""
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    log = []
    
    for step in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(input_ids)
        next_token_logits = outputs.logits[0, -1]
        probs = F.softmax(next_token_logits, dim=-1)
        topk_probs, topk_ids = probs.topk(k)
        
        next_id = topk_ids[0].item()
        log.append({
            "step": step,
            "chosen_token": tokenizer.decode([next_id]),
            "chosen_prob": topk_probs[0].item(),
            "top_k": [(tokenizer.decode([i.item()]), p.item()) for p, i in zip(topk_probs, topk_ids)]
        })
        
        input_ids = torch.cat([input_ids, topk_ids[:1].unsqueeze(0)], dim=1)
        if next_id == tokenizer.eos_token_id:
            break
    
    return tokenizer.decode(input_ids[0]), log


text, log = greedy_decode_with_topk_log(model, tokenizer, prompt, max_new_tokens=15)
for entry in log:
    bar = "█" * int(entry["chosen_prob"] * 20)
    print(f"step {entry['step']:2d}  p={entry['chosen_prob']:.3f} {bar:<20}  '{entry['chosen_token']}'")

print()
print("full text:", text)

## Reflection questions

Before moving on, write down your answers to these in `docs/concepts/01_decoding_basics.md`:

1. The forward pass produces a logit distribution at **every** position, not just the last. Why don't we use the others during decoding? (Hint: causal masking + we already know those tokens.)
2. What is the relationship between `chosen_prob` in the log above and the model's *uncertainty*?
3. Suppose a generation has 50 tokens, and 40 of them have `chosen_prob > 0.9`. What does that suggest about how much of the model's compute was "necessary"?
4. Why is this loop so slow? Where exactly is the wasted work?

Questions 3 and 4 are the entire motivation for the project. Hold them in mind.